# Semi-Supervised Classification with Graph Convolution Networks

* 사용 데이터 셋 : gemsec facebook dataset

## 1. Libraries

In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def set_seed(seed=42):
    # 1. 파이썬 기본 랜덤 시드 고정
    random.seed(seed)
    
    # 2. 넘파이 시드 고정
    np.random.seed(seed)
    
    # 3. 파이토치 시드 고정 (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # 멀티 GPU 사용 시
    
    # 4. CuDNN 결정론적 연산 설정 (속도는 약간 느려질 수 있으나 재현성 보장)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # 5. OS 환경 변수 고정 (일부 라이브러리 영향)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    print(f"Seed fixed to: {seed}")

# 실행
set_seed(42)

Seed fixed to: 42


## 2. Load Data

In [3]:
# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# 1. 클래스 매핑 (8개)
class_map = {
    'artist': 0, 'athletes': 1, 'company': 2, 'government': 3,
    'new_sites': 4, 'politician': 5, 'public_figure': 6, 'tvshow': 7
}

all_edge_list = []
node_to_label = {}

# 2. 각 파일 읽기
for class_name, class_idx in class_map.items():
    df = pd.read_csv(f'E:/Study/수업/대학원/4-2/다변수 네트워크/논문리뷰/gemsec_facebook_dataset/facebook_clean_data/{class_name}_edges.csv')
    
    # 간선 리스트 추가
    edges = torch.tensor(df[['node_1', 'node_2']].values, dtype=torch.long)
    all_edge_list.append(edges)
    
    # 노드 라벨 할당
    unique_nodes = np.unique(df.values)
    for node in unique_nodes:
        node_to_label[node] = class_idx


# 전체 에지 및 노드 수 파악
edge_index = torch.cat(all_edge_list, dim=0).t() # [2, E]
num_nodes = max(node_to_label.keys()) + 1


# 라벨 텐서 생성
y = torch.zeros(num_nodes, dtype=torch.long)
for node, label in node_to_label.items():
    y[node] = label

## 3. GCN 정규화 인접 행렬 (Sparse) 계산

In [5]:
# --- 1단계: adj_norm 생성 및 GPU 이동 ---
# A_tilde = A + I
loop_indices = torch.arange(0, num_nodes).unsqueeze(0).repeat(2, 1)
adj_indices = torch.cat([edge_index, loop_indices], dim=1)
adj_values = torch.ones(adj_indices.shape[1])

# Degree 계산 (D_tilde)
row_idx, col_idx = adj_indices
deg_temp = torch.zeros(num_nodes)
deg_temp.scatter_add_(0, row_idx, adj_values)

# D^-1/2 계산
deg_inv_sqrt = deg_temp.pow(-0.5)
deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.

# 정규화 연산 및 Sparse 텐서 생성
adj_values_norm = deg_inv_sqrt[row_idx] * adj_values * deg_inv_sqrt[col_idx]
# 생성 시점에 바로 .to(device)를 호출하여 GPU로 보냅니다.
adj_norm = torch.sparse_coo_tensor(adj_indices, adj_values_norm, (num_nodes, num_nodes)).to(device)

# --- 2단계: LDP (Local Degree Profile) 특징 생성 ---
# edge_index의 각 행을 GPU로 보냅니다.
row = edge_index[0].to(device)
col = edge_index[1].to(device)

# 1. 각 노드의 실제 degree 계산 (GPU 상에서 수행)
deg = torch.zeros(num_nodes).to(device)
deg.scatter_add_(0, row, torch.ones(row.size(0)).to(device))

# 2. 이웃의 degree 정보 취합 (SPMM 연산)
# 이제 adj_norm과 deg 모두 같은 device에 있으므로 에러가 발생하지 않습니다.
deg_neighbor_avg = torch.spmm(adj_norm, deg.unsqueeze(1)) 

# 3. 여러 통계량을 합쳐서 특징 행렬 구성 (Degree + Neighbor Average Degree)
# 두 텐서가 모두 GPU에 있으므로 concat 후 바로 사용 가능합니다.
features_raw = torch.cat([deg.unsqueeze(1), deg_neighbor_avg], dim=1)

# Linear 레이어를 통해 128차원으로 확장 (가중치도 GPU에 있어야 함)
projection = torch.nn.Linear(2, 128).to(device)
with torch.no_grad():
    features = projection(features_raw)

print(f"Features Device: {features.device}")
print(f"Adj_norm Device: {adj_norm.device}")

Features Device: cuda:0
Adj_norm Device: cuda:0


## 4. Model

In [6]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_norm):
        # 1. Linear Transformation (X * W)
        # 만약 X가 Sparse라면 torch.spmm 사용, Dense라면 torch.mm 사용
        support = torch.mm(x, self.weight)
        
        # 2. Graph Aggregation (A_hat * support)
        # adj_norm은 무조건 Sparse이므로 torch.spmm 사용
        output = torch.spmm(adj_norm, support)
        return output

class GCN(nn.Module):
    def __init__(self, n_feat, n_hid, n_class, dropout):
        super(GCN, self).__init__()
        self.gc1 = GCNLayer(n_feat, n_hid)
        self.gc2 = GCNLayer(n_hid, n_class)
        self.dropout = dropout

    def forward(self, x, adj_norm):
        x = F.relu(self.gc1(x, adj_norm))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj_norm)
        return F.log_softmax(x, dim=1)

## 5. Masking

In [7]:
# stratified : class 별로 샘플 갯수 차이가 크기때문에 과적합 현상이 발생
# 따라서 class 별로 동일한 갯수의 샘플 수를 뽑아 학습 진행

def create_fixed_sample_mask(y, num_nodes, samples_per_class=500):
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    
    for c in range(8): # 8개 카테고리
        # 해당 클래스의 전체 인덱스 추출
        class_indices = (y == c).nonzero(as_tuple=False).view(-1)
        
        # 셔플
        perm = torch.randperm(class_indices.size(0))
        class_indices = class_indices[perm]
        
        # 고정된 숫자만큼 할당
        n_train = min(samples_per_class, len(class_indices))
        n_val = min(samples_per_class, len(class_indices) - n_train)
        
        train_mask[class_indices[:n_train]] = True
        val_mask[class_indices[n_train:n_train + n_val]] = True
        
    # 나머지는 모두 테스트용
    test_mask = ~(train_mask | val_mask)
    
    return train_mask, val_mask, test_mask

# 클래스당 50개씩 샘플링 (총 400개 학습 데이터)
train_mask, val_mask, test_mask = create_fixed_sample_mask(y, num_nodes, samples_per_class=500)

print(f"Train nodes: {train_mask.sum().item()} (All classes balanced)")

Train nodes: 2000 (All classes balanced)


## 6. Train

In [8]:
# 1. 데이터 이동 (라벨 및 인덱스)
y = y.to(device)

# 2. Sparse 행렬 이동
# Sparse 행렬도 .to(device)를 통해 GPU 메모리로 올릴 수 있습니다.
adj_norm = adj_norm.to(device)

In [9]:
model = GCN(n_feat=128, n_hid=32, n_class=8, dropout=0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

In [10]:
# 체크포인트 저장 설정
checkpoint_path = 'E:/Study/수업/대학원/4-2/다변수 네트워크/논문리뷰/gemsec_facebook_dataset/weight3'
if not os.path.exists(checkpoint_path):
    os.makedirs(checkpoint_path)

history = [] # 학습 로그 저장
best_val_acc = 0.0
save_interval = 30 

for epoch in range(300):
    # --- Training ---
    model.train()
    optimizer.zero_grad()
    
    logits = model(features, adj_norm)  # Forward pass: 전체 그래프 정보 활용
    loss = F.nll_loss(logits[train_mask], y[train_mask])  
    loss.backward()
    optimizer.step()
    
    # --- Evaluation ---
    model.eval()
    with torch.no_grad():
        val_logits = model(features, adj_norm)
        val_loss = F.nll_loss(val_logits[val_mask], y[val_mask])
        val_pred = val_logits[val_mask].max(1)[1]
        val_acc = val_pred.eq(y[val_mask]).sum().item() / val_mask.sum().item()
        
        train_pred = logits[train_mask].max(1)[1]
        train_acc = train_pred.eq(y[train_mask]).sum().item() / train_mask.sum().item()

    # 기록 저장 (CPU 값으로 변환하여 리스트에 추가)
    history.append({
        'epoch': epoch+1,
        'loss': loss.item(),
        'val_loss': val_loss.item(),
        'train_acc': train_acc,
        'val_acc': val_acc
    })

    # 10 에포크마다 출력 및 CSV 중간 저장
    # 10 에포크마다 출력 (val_loss 포함)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {(epoch+1):03d} | Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        # 중간 기록 저장 (학습 도중 중단 대비)
        df_history = pd.DataFrame(history)
        df_history.to_csv(f'{checkpoint_path}/learning_history.csv', index=False)

    # save_interval 마다 가중치 저장
    if epoch % save_interval == 0 and epoch != 0:
        # 에포크 번호를 파일명에 포함하여 저장
        torch.save({

            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, f'{checkpoint_path}/gcn_weights_epoch_{epoch}.pth')
        print(f"==> 에포크 {epoch}: 가중치 체크포인트 저장 완료")

# 4. 최종 결과 저장
df_history = pd.DataFrame(history)
df_history.to_csv(f'{checkpoint_path}/final_learning_history.csv', index=False)
print(f"Training finished. History saved to {checkpoint_path}/final_learning_history.csv")

Epoch 010 | Loss: 11.6933 | Val Loss: 10.1310 | Train Acc: 0.2265 | Val Acc: 0.2415
Epoch 020 | Loss: 3.7307 | Val Loss: 3.1937 | Train Acc: 0.2700 | Val Acc: 0.2505
Epoch 030 | Loss: 2.7733 | Val Loss: 1.9649 | Train Acc: 0.3045 | Val Acc: 0.2610
==> 에포크 30: 가중치 체크포인트 저장 완료
Epoch 040 | Loss: 1.8221 | Val Loss: 1.2392 | Train Acc: 0.3725 | Val Acc: 0.4675
Epoch 050 | Loss: 1.6801 | Val Loss: 1.1109 | Train Acc: 0.3960 | Val Acc: 0.5355
Epoch 060 | Loss: 1.4754 | Val Loss: 1.0535 | Train Acc: 0.4550 | Val Acc: 0.5470
==> 에포크 60: 가중치 체크포인트 저장 완료
Epoch 070 | Loss: 1.3932 | Val Loss: 0.9958 | Train Acc: 0.4725 | Val Acc: 0.6050
Epoch 080 | Loss: 1.2757 | Val Loss: 0.9357 | Train Acc: 0.4905 | Val Acc: 0.6175
Epoch 090 | Loss: 1.1753 | Val Loss: 0.9114 | Train Acc: 0.5320 | Val Acc: 0.6355
==> 에포크 90: 가중치 체크포인트 저장 완료
Epoch 100 | Loss: 1.1106 | Val Loss: 0.8645 | Train Acc: 0.5510 | Val Acc: 0.6635
Epoch 110 | Loss: 1.0822 | Val Loss: 0.8534 | Train Acc: 0.5570 | Val Acc: 0.6655
Epoch 120 | 